# YuE2 — standard ComfyUI canvas

Attach your existing **yue2-music-assets**, choose **GPU T4 x2**, and enable
**Internet**. Run the one code cell below. It installs the ComfyUI interface and
prints two Cloudflare links: **Song Canvas** and **WAV → ABC Canvas**. Nothing
runs until you press **Run** inside ComfyUI. There is no AIQUEST/Gradio wrapper.

The song canvas has separate, connected prompt, settings, planning, semantic,
acoustic, decoding, saving and standard audio-preview nodes. Edit the lyrics/style
and settings directly on those nodes. Native FP16/CUDA-graph generation, T4
attention patches, 16 Euler steps, CFG 1 and the 8-minute safety ceiling remain.
ABC planning has its own visible `abc_token_limit` control (default 4096);
it is independent of the song ceiling and unused in direct/off mode.
The model can end earlier naturally. Saved status distinguishes a natural ending
from a safety cutoff. Long-song quality, runtime and GPU peaks remain unverified.

For a cover/ABC transcription, open the **WAV → ABC** canvas, use ComfyUI's normal
**Load Audio** upload button, then Run. It displays the extracted approximate
melody ABC on the node. Analysis artifacts and score are saved in cover_sources.
You can paste that ABC into the song prompt node's ABC field, select melody
planning, and generate with your new style/lyrics. No lossy conversion is needed.

Your existing model dataset is reused without rebuilding. The new ComfyUI source,
frontend and server dependencies require a first-session Internet installation;
this is additional interface software, not another model download. It uses a
separate environment so the working native runtime is preserved. Comfy's controller
runs on CPU; native nodes explicitly use T4 0 for MoT and T4 1 for VAE as before.

Songs, tokens, latents and metadata stay in `/kaggle/working/yue2_native/runs`.
Use ComfyUI's audio preview/download and Save Version with outputs to preserve
songs before ending the session. Keep the code cell running while using the canvas.
The old working notebook remains available as a fallback. This interface still
requires live Kaggle validation; CPU/schema checks do not prove GPU generation.


In [ ]:
import base64, json, os, signal, subprocess, sys, zlib
from pathlib import Path
SUPPORT=Path('/kaggle/temp/yue2_comfy_support')
ROOT='/kaggle/temp/yue2_native'
OUTPUTS='/kaggle/working/yue2_native/runs'
ASSET_DATASET='auto'
payload="..."
for relative,contents in json.loads(zlib.decompress(base64.b64decode(payload))).items():
    path=SUPPORT/relative
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(contents)
env=dict(os.environ)
env['PYTHONPATH']=str(SUPPORT)+os.pathsep+env.get('PYTHONPATH','')
env['PYTHONUNBUFFERED']='1'
log_path=Path(OUTPUTS).parent/'runtime/comfy-startup.log'
log_path.parent.mkdir(parents=True,exist_ok=True)
worker=subprocess.Popen([sys.executable,'-u','-m','native_yue2.comfy',
    '--root',ROOT,'--outputs',OUTPUTS,'--assets',ASSET_DATASET],env=env,
    stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,start_new_session=True)
try:
    with log_path.open('a') as log:
        for line in worker.stdout:
            log.write(line);log.flush()
            print(line,end='',flush=True)
    if worker.wait():raise RuntimeError(f'ComfyUI startup stopped. Full log: {log_path}')
finally:
    if worker.poll() is None:
        os.killpg(worker.pid,signal.SIGTERM)
        try:worker.wait(timeout=20)
        except subprocess.TimeoutExpired:
            os.killpg(worker.pid,signal.SIGKILL);worker.wait(timeout=5)
